# 30-Minute CGM Glucose Forecasting
A university project comparing MLP and LSTM models with persistence using nested cross-patient validation. Historical outputs are archived separately from the executable protocol. No new training results are claimed.

## Dataset
The original arrays contain 12 patient sequences: training shape (12, 12641, 2), additional test shape (12, 2897, 2). Each row contains a timestamp and glucose in mg/dL. The source and distribution permissions are not documented. No raw records or individual glucose trajectories are distributed. Patient numbers below are array positions, not identities. LOPO uses train_series only; the additional test array is loaded but not used for the reported evaluation.

## Environment and protocol selection
Original output confirms TensorFlow 2.19.0. Install requirements.txt in Python 3.10–3.12. Other original versions were not recorded. Seeds are preserved without a guarantee of numerical determinism.

Choose `corrected` (default) for gap-aware windowing and patient-separated final validation, or `historical` to retain the original computational choices. Corrected results require a new full run and must not be labelled with the saved historical metrics.


In [ ]:
import os
PROTOCOL = "corrected"  # Alternative: "historical"
assert PROTOCOL in {"corrected", "historical"}
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataclasses import dataclass
from typing import Dict, Tuple, List

from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TF version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# Optional Colab support:
# from google.colab import drive
# drive.mount('/content/drive')
BASE_PATH = os.environ.get('CGM_DATASET_ROOT', 'data')
TRAIN_PATH = os.path.join(BASE_PATH, 'glucose_level.npy')
TEST_PATH = os.path.join(BASE_PATH, 'glucose_level_test.npy')
# Object arrays require pickle: load only trusted local files.
train_raw = np.load(TRAIN_PATH, allow_pickle=True)
test_raw = np.load(TEST_PATH, allow_pickle=True) if os.path.exists(TEST_PATH) else None
print('Training array shape:', train_raw.shape)


## Signal preprocessing and missing values
Historical parsing anchors a regular grid at the first timestamp, reindexes exactly, interpolates up to six missing values even in longer gaps, and drops remaining NaNs. Off-grid observations may be discarded. The corrected parser retains all original timestamps for interpolation, fills only bounded runs of at most six missing grid samples and preserves NaNs over longer gaps. The regular output grid remains anchored at the first observation; off-grid observations act as interpolation anchors, not as rounded measurements.

Time interpolation is retrospective: a later anchor can influence earlier imputed input values. This remaining look-ahead risk requires a causal preprocessing study before prospective use. Six missing samples denotes 30 minutes of missing grid points; bounding observations may be 35 minutes apart.


In [ ]:
def parse_patient_array_historical(patient_arr: np.ndarray,
                        freq: str = "5min",
                        time_format: str = "%d-%m-%Y %H:%M:%S",
                        interpolate: bool = True,
                        max_gap_steps: int = 6) -> pd.Series:

    df = pd.DataFrame(patient_arr, columns=["timestamp", "glucose"])

    df["timestamp"] = pd.to_datetime(df["timestamp"], format=time_format, errors="coerce")
    df["glucose"]   = pd.to_numeric(df["glucose"], errors="coerce")

    df = df.dropna(subset=["timestamp", "glucose"]).sort_values("timestamp")

    # Remove duplicates (keep last)
    df = df.drop_duplicates(subset=["timestamp"], keep="last")
    df = df.set_index("timestamp")

    # Uniform grid (5 minutes)
    full_index = pd.date_range(df.index.min(), df.index.max(), freq=freq)
    s = df["glucose"].reindex(full_index)

    if interpolate:
        s = s.interpolate(method="time", limit=max_gap_steps)
        s = s.dropna()
    return s.astype("float32")



In [ ]:
def parse_patient_array_corrected(patient_arr, freq="5min", time_format="%d-%m-%Y %H:%M:%S", interpolate=True, max_gap_steps=6):
    df = pd.DataFrame(patient_arr, columns=['timestamp', 'glucose'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], format=time_format, errors='coerce')
    df['glucose'] = pd.to_numeric(df['glucose'], errors='coerce')
    df = df.dropna().sort_values('timestamp').drop_duplicates('timestamp', keep='last')
    if df.empty:
        return pd.Series([], index=pd.DatetimeIndex([]), dtype='float32')
    raw = df.set_index('timestamp')['glucose'].astype(float)
    grid = pd.date_range(raw.index.min(), raw.index.max(), freq=freq)
    observed = raw.reindex(grid)
    if not interpolate:
        return observed.astype('float32')
    missing = observed.isna()
    runs = missing.ne(missing.shift()).cumsum()
    lengths = missing.groupby(runs).transform('sum')
    estimates = raw.reindex(raw.index.union(grid)).interpolate(method='time', limit_area='inside').reindex(grid)
    eligible = missing & (lengths <= max_gap_steps)
    return observed.where(~eligible, estimates).astype('float32')

parse_patient_array = parse_patient_array_corrected if PROTOCOL == 'corrected' else parse_patient_array_historical
train_series = [parse_patient_array(x) for x in train_raw]
# test_raw is not used by the LOPO experiment.


## Sliding-window generation
Configuration: 24 input samples, horizon 6 and step 1. Inputs have shape (N, 24, 1), targets (N, 1). The target at i+24+6−1 is exactly 30 minutes after the final input sample. The 24 observations span 115 minutes between endpoints and represent a two-hour sampling block. Historical windowing omits the last valid start and ignores timestamps after NaN removal. Corrected windowing includes the last start and requires the entire input-to-target interval to be finite and spaced at five minutes.


In [ ]:
@dataclass
class ForecastConfig:
    input_len: int      # Number of past points to use as features
    horizon: int        # Forecasting horizon (predicting value at t + horizon)
    step: int = 1       # Shift for the sliding window

# Initial configuration (2 hours of history to predict 30 mins)
cfg = ForecastConfig(input_len=24, horizon=6, step=1)
print(cfg)


def make_windows_historical(series: pd.Series, cfg: ForecastConfig) -> Tuple[np.ndarray, np.ndarray]:
    x = series.values.astype("float32")
    X_list, y_list = [], []

    L = cfg.input_len
    H = cfg.horizon
    step = cfg.step

    # Define the last possible starting index to keep the horizon within bounds
    last = len(x) - (L + H)

    # Slide the window
    for i in range(0, last, step):
        X_list.append(x[i:i+L])
        # Target: the single point located exactly H steps after the window ends
        y_list.append(x[i+L+H-1])

    # Convert to numpy arrays and add the feature/channel dimension for Keras (N, L, 1)
    X = np.array(X_list, dtype="float32")[..., None]
    y = np.array(y_list, dtype="float32")[..., None]
    return X, y




In [ ]:
def make_windows_corrected(series, cfg):
    if min(cfg.input_len, cfg.horizon, cfg.step) < 1:
        raise ValueError('Window configuration must be positive')
    x = series.to_numpy(dtype='float32')
    X, y = [], []
    width = cfg.input_len + cfg.horizon
    for i in range(0, len(x) - width + 1, cfg.step):
        span = x[i:i+width]
        times = series.index[i:i+width]
        if not np.isfinite(span).all() or not (np.diff(times.asi8) == pd.Timedelta('5min').value).all():
            continue
        X.append(span[:cfg.input_len])
        y.append(span[-1])
    return np.asarray(X, dtype='float32').reshape(-1, cfg.input_len, 1), np.asarray(y, dtype='float32').reshape(-1, 1)

make_windows = make_windows_corrected if PROTOCOL == 'corrected' else make_windows_historical


## Evaluation metrics

In [ ]:
def rmse(y_true, y_pred) -> float:
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mape_safe(y_true, y_pred, eps: float = 1.0) -> float:

    y_true = y_true.reshape(-1)
    y_pred = y_pred.reshape(-1)
    denom = np.maximum(np.abs(y_true), eps)
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

def compute_metrics(y_true, y_pred) -> Dict[str, float]:
    # Flatten inputs and return a dictionary of evaluation metrics
    y_true = y_true.reshape(-1)
    y_pred = y_pred.reshape(-1)
    return {
        "MAE":  float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(rmse(y_true, y_pred)),
        "MAPE": float(mape_safe(y_true, y_pred, eps=1.0)),
    }

## Training-fold standardisation

In [ ]:
@dataclass
class Standardizer:
    mean_: float = 0.0
    std_: float = 1.0

    def fit(self, x: np.ndarray):
        self.mean_ = float(np.mean(x))
        # Add a tiny epsilon (1e-8) to prevent division by zero during transformation
        self.std_  = float(np.std(x) + 1e-8)
        return self

    def transform(self, x: np.ndarray) -> np.ndarray:
        return (x - self.mean_) / self.std_

    def inverse(self, x: np.ndarray) -> np.ndarray:
        return x * self.std_ + self.mean_

## Persistence baseline

In [ ]:
def baseline_persistence_predict(X: np.ndarray) -> np.ndarray:
    return X[:, -1, :]

## Multi-layer perceptron

In [ ]:
def build_mlp(input_len: int, lr: float = 1e-3) -> keras.Model:
    inp = keras.Input(shape=(input_len, 1))
    x = layers.Flatten()(inp)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(1)(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="mse")
    return model

## Long short-term memory model

In [ ]:
def build_lstm(input_len: int, lr: float = 1e-3) -> keras.Model:
    inp = keras.Input(shape=(input_len, 1))
    x = layers.LSTM(64, return_sequences=False)(inp)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation="relu")(x)
    out = layers.Dense(1)(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="mse")
    return model

## Training and early stopping

In [ ]:
def make_tf_dataset(X, y, batch_size=256, training=True):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        ds = ds.shuffle(min(len(X), 5000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


def fit_model(model, Xtr, ytr, Xva, yva, batch=256, epochs=30, patience=3, verbose=0):
    ds_tr = make_tf_dataset(Xtr, ytr, batch_size=batch, training=True)
    ds_va = make_tf_dataset(Xva, yva, batch_size=batch, training=False)

    cb = keras.callbacks.EarlyStopping(monitor="val_loss",
                                       patience=patience,
                                       restore_best_weights=True)
    hist = model.fit(ds_tr, validation_data=ds_va,
                     epochs=epochs, callbacks=[cb], verbose=verbose)
    return hist

## Hyperparameter configurations

In [ ]:
param_grid = [
    {"model_type": "MLP",  "lr": 1e-3,  "batch": 256},
    {"model_type": "MLP",  "lr": 3e-4,  "batch": 256},
    {"model_type": "LSTM", "lr": 1e-3,  "batch": 256},
]

def build_model_by_type(model_type: str, input_len: int, lr: float) -> keras.Model:

    if model_type == "MLP":
        return build_mlp(input_len, lr=lr)
    elif model_type == "LSTM":
        return build_lstm(input_len, lr=lr)
    else:
        raise ValueError("Unknown model_type")

## Patient grouping

In [ ]:
def stack_patients_windows(patient_ids: List[int], series_list: List[pd.Series], cfg: ForecastConfig):
    X_all, y_all, g_all = [], [], []
    for pid in patient_ids:
        Xp, yp = make_windows(series_list[pid], cfg)
        X_all.append(Xp); y_all.append(yp)
        g_all.append(np.full((len(Xp),), pid))
    X_all = np.concatenate(X_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)
    g_all = np.concatenate(g_all, axis=0)
    return X_all, y_all, g_all

## Inner GroupKFold selection

In [ ]:
def inner_cv_select_params(train_pids: List[int],
                           series_list: List[pd.Series],
                           cfg: ForecastConfig,
                           param_grid: List[dict],
                           n_splits: int = 5,
                           epochs: int = 5,
                           patience: int = 1,
                           verbose_fit: int = 0) -> dict:

    X, y, groups = stack_patients_windows(train_pids, series_list, cfg)

    gkf = GroupKFold(n_splits=n_splits)
    results = []

    for params in param_grid:
        fold_metrics = []

        for tr_idx, va_idx in gkf.split(X, y, groups=groups):
            Xtr, ytr = X[tr_idx], y[tr_idx]
            Xva, yva = X[va_idx], y[va_idx]

            # Standardize data using statistics calculated only from the current training fold
            # This prevents "data leakage" from the validation set
            scaler = Standardizer().fit(Xtr.reshape(-1))
            Xtr_s = scaler.transform(Xtr)
            Xva_s = scaler.transform(Xva)

            # Build & train
            model = build_model_by_type(params["model_type"], cfg.input_len, params["lr"])
            _ = fit_model(model, Xtr_s, ytr, Xva_s, yva,
                          batch=params["batch"], epochs=epochs,
                          patience=patience, verbose=verbose_fit)

            # Predict
            yhat = model.predict(Xva_s, verbose=0)

            m = compute_metrics(yva, yhat)
            fold_metrics.append(m)

            keras.backend.clear_session()

        # Aggregate fold metrics
        mean_rmse = float(np.mean([fm["RMSE"] for fm in fold_metrics]))
        mean_mae  = float(np.mean([fm["MAE"]  for fm in fold_metrics]))
        mean_mape = float(np.mean([fm["MAPE"] for fm in fold_metrics]))

        results.append((mean_rmse, mean_mae, mean_mape, params))

        print(f"[InnerCV] {params} -> mean RMSE={mean_rmse:.3f}, mean MAE={mean_mae:.3f}, mean MAPE={mean_mape:.2f}%")

    # Choose best by RMSE
    results.sort(key=lambda x: x[0])
    best = results[0][3]
    print("\nBest params:", best)
    return best

## Leave-one-patient-out evaluation

In [ ]:
def train_and_eval_on_patient(train_pids: List[int],
                              test_pid: int,
                              series_list: List[pd.Series],
                              cfg: ForecastConfig,
                              best_params: dict,
                              epochs: int = 30,
                              patience: int = 3,
                              verbose_fit: int = 0):



    # --- Build training windows (all training patients) ---
    Xtr, ytr, _ = stack_patients_windows(train_pids, series_list, cfg)

    # --- Build test windows (left-out patient) ---
    Xte, yte = make_windows(series_list[test_pid], cfg)

    # --- Baseline: persistence on the same test windows ---
    # Baseline predicts the last observed value in each input window
    y_base = baseline_persistence_predict(Xte).reshape(-1)   # (N,)

    if PROTOCOL == 'historical':
        # --- Standardize inputs using ONLY training data (no leakage) ---
        scaler = Standardizer().fit(Xtr.reshape(-1))
        Xtr_s = scaler.transform(Xtr)
        Xte_s = scaler.transform(Xte)
    
        # --- Build model ---
        model = build_model_by_type(best_params["model_type"], cfg.input_len, best_params["lr"])
    
        # --- Small validation split ONLY from training windows (for early stopping) ---
        n = len(Xtr_s)
        idx = np.arange(n)
        rng = np.random.default_rng(SEED)  # reproducible
        rng.shuffle(idx)
    
        split = int(0.9 * n)
        tr_idx, va_idx = idx[:split], idx[split:]
    
    else:
        # Hold out complete training patients for final early stopping.
        Xtr, ytr, groups = stack_patients_windows(train_pids, series_list, cfg)
        if len(Xte) == 0:
            raise ValueError('Held-out patient has no valid corrected windows')
        splitter = GroupKFold(n_splits=5)
        tr_idx, va_idx = next(splitter.split(Xtr, ytr, groups))
        scaler = Standardizer().fit(Xtr[tr_idx].reshape(-1))
        Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)
        model = build_model_by_type(best_params['model_type'], cfg.input_len, best_params['lr'])

    # --- Train ---
    _ = fit_model(
        model,
        Xtr_s[tr_idx], ytr[tr_idx],
        Xtr_s[va_idx], ytr[va_idx],
        batch=best_params["batch"],
        epochs=epochs,
        patience=patience,
        verbose=verbose_fit
    )

    # --- Predict on left-out patient ---
    yhat = model.predict(Xte_s, verbose=0)

    # --- Compute metrics (MODEL only) ---
    metrics = compute_metrics(yte, yhat)

    # --- Flatten for plotting convenience ---
    y_true_1d = yte.reshape(-1)
    y_pred_1d = yhat.reshape(-1)
    y_base_1d = y_base.reshape(-1)

    keras.backend.clear_session()
    return metrics, y_true_1d, y_pred_1d, y_base_1d


def run_lopo_cv(series_list: List[pd.Series],
                cfg: ForecastConfig,
                param_grid: List[dict],
                inner_splits: int = 5,
                verbose_fit: int = 0):

    all_pids = list(range(len(series_list)))

    outer_metrics = []
    chosen_params = []

    # --- Store predictions for final plots ---
    all_y_true = []     # list of 1D arrays (per patient)
    all_y_pred = []     # list of 1D arrays (per patient)
    all_y_base = []     # list of 1D arrays (per patient)

    for test_pid in all_pids:
        train_pids = [p for p in all_pids if p != test_pid]

        print("\n" + "=" * 70)
        print(f"OUTER LOOP — Test patient: {test_pid} | Train patients: {train_pids}")
        print("=" * 70)

        # --- Inner CV: hyperparameter selection on training patients only ---
        best = inner_cv_select_params(
            train_pids=train_pids,
            series_list=series_list,
            cfg=cfg,
            param_grid=param_grid,
            n_splits=inner_splits,
            epochs=15,
            patience=2,
            verbose_fit=verbose_fit
        )
        chosen_params.append(best)

        # --- Outer test: train on train_pids, evaluate on test_pid ---
        m, y_true_pid, y_pred_pid, y_base_pid = train_and_eval_on_patient(
            train_pids=train_pids,
            test_pid=test_pid,
            series_list=series_list,
            cfg=cfg,
            best_params=best,
            epochs=25,
            patience=3,
            verbose_fit=verbose_fit
        )

        print(
            f"[OuterTest] Patient {test_pid} -> "
            f"MAE={m['MAE']:.3f}, RMSE={m['RMSE']:.3f}, MAPE={m['MAPE']:.2f}%"
        )

        outer_metrics.append(m)
        all_y_true.append(y_true_pid)
        all_y_pred.append(y_pred_pid)
        all_y_base.append(y_base_pid)

    # --- Summary mean ± std across patients ---
    mae_vals  = np.array([m["MAE"]  for m in outer_metrics], dtype=float)
    rmse_vals = np.array([m["RMSE"] for m in outer_metrics], dtype=float)
    mape_vals = np.array([m["MAPE"] for m in outer_metrics], dtype=float)

    summary = {
        "MAE_mean":  float(mae_vals.mean()),
        "MAE_std":   float(mae_vals.std(ddof=1)),
        "RMSE_mean": float(rmse_vals.mean()),
        "RMSE_std":  float(rmse_vals.std(ddof=1)),
        "MAPE_mean": float(mape_vals.mean()),
        "MAPE_std":  float(mape_vals.std(ddof=1)),
    }

    return outer_metrics, summary, chosen_params, all_y_true, all_y_pred, all_y_base


# =========================
# Run LOPO
# =========================
outer_metrics, summary, chosen_params, all_y_true, all_y_pred, all_y_base = run_lopo_cv(
    series_list=train_series,
    cfg=cfg,
    param_grid=param_grid,
    inner_splits=5,
    verbose_fit=0
)

print("\n" + "#"*70)
print("FINAL RESULTS (Leave-One-Patient-Out) — mean ± std")
print("#"*70)
print(f"MAE  = {summary['MAE_mean']:.3f} ± {summary['MAE_std']:.3f}  mg/dL")
print(f"RMSE = {summary['RMSE_mean']:.3f} ± {summary['RMSE_std']:.3f} mg/dL")
print(f"MAPE = {summary['MAPE_mean']:.3f} ± {summary['MAPE_std']:.3f}  %")

## Patient-level performance

In [ ]:
df_patients = pd.DataFrame(outer_metrics)

# Ensure patient id
df_patients["Patient"] = np.arange(len(df_patients))

# Reorder and rename columns
df_patients = df_patients[["Patient", "MAE", "RMSE", "MAPE"]]

df_patients = df_patients.rename(columns={
    "Patient": "Patient ID",
    "MAE": "MAE (mg/dL)",
    "RMSE": "RMSE (mg/dL)",
    "MAPE": "MAPE (%)"
})

# Round values for nicer display
df_patients = df_patients.round({
    "MAE (mg/dL)": 2,
    "RMSE (mg/dL)": 2,
    "MAPE (%)": 2
})

df_patients

## Patient-level error plot

In [ ]:
df_patients.plot(
    x="Patient ID",
    y=["MAE (mg/dL)", "RMSE (mg/dL)"],
    kind="bar",
    figsize=(10,4)
)

plt.ylabel("mg/dL")
plt.title("Patient-level performance (Outer Test – LOPO)")
plt.tight_layout()
plt.show()

## Persistence comparison

In [ ]:
rows = []

for pid in range(len(all_y_true)):
    # Ground truth and baseline aligned by LOPO
    y_true = np.asarray(all_y_true[pid]).reshape(-1)
    y_base = np.asarray(all_y_base[pid]).reshape(-1)

    # Safety check
    assert len(y_true) == len(y_base)
    n = len(y_true)
    if n == 0:
        continue

    y_true = y_true[:n]
    y_base = y_base[:n]

    m = compute_metrics(y_true, y_base)

    rows.append({
        "Patient ID": pid,
        "MAE_base":  m["MAE"],
        "RMSE_base": m["RMSE"],
        "MAPE_base": m["MAPE"],
    })

# Patient-level baseline table
df_baseline_lopo = pd.DataFrame(rows)  # Keep full precision for new summaries
df_baseline_lopo

print("-" * 60)
print("REFERENCE BASELINE: PERSISTENCE (LOPO-consistent)")
print("-" * 60)
print(f"MAE  = {df_baseline_lopo['MAE_base'].mean():.3f} ± {df_baseline_lopo['MAE_base'].std(ddof=1):.3f} mg/dL")
print(f"RMSE = {df_baseline_lopo['RMSE_base'].mean():.3f} ± {df_baseline_lopo['RMSE_base'].std(ddof=1):.3f} mg/dL")
print(f"MAPE = {df_baseline_lopo['MAPE_base'].mean():.3f} ± {df_baseline_lopo['MAPE_base'].std(ddof=1):.3f} %")
print("-" * 60)


## Error versus glucose variation

In [ ]:
# Collect all points across patients
delta_all = []
err_model_all = []
err_base_all = []

for pid in range(len(all_y_true)):
    y_true = np.asarray(all_y_true[pid]).reshape(-1)
    y_pred = np.asarray(all_y_pred[pid]).reshape(-1)
    y_base = np.asarray(all_y_base[pid]).reshape(-1)

    assert len(y_true) == len(y_pred) == len(y_base)
    n = len(y_true)
    y_true, y_pred, y_base = y_true[:n], y_pred[:n], y_base[:n]

    delta = np.abs(y_true - y_base)          # glucose variation
    err_model = np.abs(y_true - y_pred)
    err_base  = np.abs(y_true - y_base)

    delta_all.append(delta)
    err_model_all.append(err_model)
    err_base_all.append(err_base)

delta_all = np.concatenate(delta_all)
err_model_all = np.concatenate(err_model_all)
err_base_all  = np.concatenate(err_base_all)

# Bin definition
bins = np.array([0, 5, 10, 20, 40, 80])
bin_centers = (bins[:-1] + bins[1:]) / 2

model_means = []
baseline_means = []

for i in range(len(bins) - 1):
    mask = (delta_all >= bins[i]) & (delta_all < bins[i+1])
    model_means.append(err_model_all[mask].mean())
    baseline_means.append(err_base_all[mask].mean())

# Plot
plt.figure(figsize=(8,4))
plt.plot(bin_centers, model_means, marker="o", lw=2, label="Model (LOPO)")
plt.plot(bin_centers, baseline_means, marker="o", lw=2, linestyle="--", label="Baseline (Persistence)")

plt.xlabel("Glucose variation |y_true − y_baseline| (mg/dL)")
plt.ylabel("Mean absolute error (mg/dL)")
plt.title("Prediction error vs glucose variation (30-min horizon)")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Historical experimental results
These values are copied from saved outputs of the historical protocol, not generated by the corrected code. Mean and sample standard deviation (ddof=1) are calculated across patients, with equal patient weighting, not pooled across predictions. The historical baseline summary was calculated after rounding patient metrics to three decimals.

| Metric | Selected neural model | Persistence |
|---|---|---|
| MAE (mg/dL) | 15.488 ± 3.612 | 17.154 ± 3.556 |
| RMSE (mg/dL) | 22.015 ± 6.552 | 23.995 ± 6.197 |
| MAPE (%) | 10.851 ± 2.784 | 11.791 ± 2.395 |

MLP with learning rate 0.001 and batch size 256 was selected in all 12 saved folds. Mean MAE decreases by 1.666 mg/dL (about 9.7%); mean RMSE by 1.980 mg/dL (about 8.3%). These are descriptive differences based on rounded summaries, not significance tests. The saved baseline output contains no per-patient table, so superiority for every patient cannot be established from the available text.


### Historical saved output
```text
    Patient ID  MAE (mg/dL)  RMSE (mg/dL)  MAPE (%)
0            0        10.67         14.45     10.57
1            1        12.70         17.61      8.15
2            2        21.30         35.31     15.30
3            3        18.76         26.10     12.80
4            4        13.13         17.40     10.44
5            5        11.44         14.45      5.90
6            6        12.86         16.81      7.75
7            7        15.50         21.20     13.12
8            8        18.09         25.37     10.05
9            9        17.44         25.13     11.79
10          10        20.47         30.42     14.39
11          11        13.52         19.92      9.93
```

### Historical saved output
```text
------------------------------------------------------------
REFERENCE BASELINE: PERSISTENCE (LOPO-consistent)
------------------------------------------------------------
MAE  = 17.154 ± 3.556 mg/dL
RMSE = 23.995 ± 6.197 mg/dL
MAPE = 11.791 ± 2.395 %
------------------------------------------------------------

```

![Historical aggregate error analysis](assets/results/original_cell_45_output_0.png)

## Prediction visualisation and privacy
Individual time series, preprocessing comparisons with calendar dates, and patient scatter plots are intentionally excluded from the release assets because data-sharing permission is undocumented. They can be generated locally after obtaining appropriate permission. The aggregate error figure above is extracted unchanged from the original output. Its x variable equals the persistence absolute error by construction: a rising persistence curve is therefore expected. It excludes variation ≥80 mg/dL and does not establish a clinical benefit during rapid change.

## Discussion and limitations
Twelve patients form a small evaluation cohort with variable usable sequence lengths. Original long-gap handling and timestamp reindexing can alter the effective forecast horizon. The final historical 90/10 random window split mixes overlapping windows and fits the scaler before the validation split; outer-test patients remain excluded. Inner scaling is correctly fitted on training folds only. Corrected mode uses the first five-fold grouped split of outer-training windows for final validation and fits its scaler on the fitting subset. This changes results and training data quantity.

Interpolation still uses future anchors retrospectively, including possible input–target dependency near missing samples. Irregular timestamp handling and gap policy need assessment with the actual data. No clinical or independent external evaluation is demonstrated. LOPO uses only the training file; the separate test file does not supply the reported test results.

## Conclusions
Saved results show lower average errors than persistence under the historical protocol. Corrected gap-aware and grouped-validation results remain to be calculated. See docs/METHODOLOGY.md for the audit and validation record.
